# 01 — Exploratory Data Analysis: Bank Customer Churn

**Dataset:** public Kaggle "Bank Customer Churn" (`Churn_Modelling.csv`) — 10,000 customers, 3 countries.
**Goal:** find *which* customers churn and *why*, so the bank can run targeted retention.
**Target:** `Churn` (1 = exited, 0 = retained).


In [ ]:
import sys
from pathlib import Path

# Make the project's src/ importable from this notebook
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src import load_data, clean_data, add_features, get_model_data

df = add_features(clean_data(load_data()))
print('Loaded:', df.shape)


## 1. Data quality check


In [ ]:
print('Rows x Cols:', df.shape)
print('Missing values:', int(df.isnull().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
df.dtypes.to_frame('dtype')


In [ ]:
df.describe(include='all').T


## 2. Overall churn rate


In [ ]:
overall = df['Churn'].mean() * 100
print(f'Overall churn rate: {overall:.2f}%  ({df["Churn"].sum():,} of {len(df):,} customers exited)')

vc = df['Churn'].value_counts().rename(index={0: 'Retained', 1: 'Exited'}).reset_index()
vc.columns = ['Outcome', 'Customers']
fig = px.bar(vc, x='Outcome', y='Customers', text='Customers',
             labels={'Outcome': '', 'Customers': 'Customers'},
             title=f'Customer outcomes — {overall:.1f}% churned')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()


## 3. Churn by geography — Germany churns at ~2x the rate


In [ ]:
g = df.groupby('Geography', observed=True)['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
g = g.reset_index().sort_values('churn_rate', ascending=False)
print(g.to_string(index=False))

fig = px.bar(g, x='Geography', y='churn_rate', text='churn_rate',
             title='Churn rate by country', labels={'churn_rate': 'Churn rate (%)', 'Geography': 'Country'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=df['Churn'].mean() * 100, line_dash='dash',
              annotation_text='Overall avg', annotation_position='top right')
fig.show()


## 4. Churn by gender


In [ ]:
g = df.groupby('Gender', observed=True)['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
g = g.reset_index().sort_values('churn_rate', ascending=False)
print(g.to_string(index=False))

fig = px.bar(g, x='Gender', y='churn_rate', text='churn_rate',
             title='Churn rate by gender', labels={'churn_rate': 'Churn rate (%)', 'Gender': 'Gender'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=df['Churn'].mean() * 100, line_dash='dash',
              annotation_text='Overall avg', annotation_position='top right')
fig.show()


## 5. Churn by age band — the 50–60 cohort is the danger zone


In [ ]:
g = df.groupby('AgeBand', observed=True)['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
g = g.reset_index().sort_values('churn_rate', ascending=False)
print(g.to_string(index=False))

fig = px.bar(g, x='AgeBand', y='churn_rate', text='churn_rate',
             title='Churn rate by age band', labels={'churn_rate': 'Churn rate (%)', 'AgeBand': 'Age band'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=df['Churn'].mean() * 100, line_dash='dash',
              annotation_text='Overall avg', annotation_position='top right')
fig.show()


## 6. Churn by number of products

**2 products is the loyalty sweet spot** (7.6% churn). One-product customers churn 3.6x more —
the classic cross-sell opportunity. (3–4 products look extreme but are tiny cohorts: n=266 and n=60.)


In [ ]:
g = df.groupby('NumOfProducts', observed=True)['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
g = g.reset_index().sort_values('churn_rate', ascending=False)
print(g.to_string(index=False))

fig = px.bar(g, x='NumOfProducts', y='churn_rate', text='churn_rate',
             title='Churn rate by products held', labels={'churn_rate': 'Churn rate (%)', 'NumOfProducts': 'Products held'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=df['Churn'].mean() * 100, line_dash='dash',
              annotation_text='Overall avg', annotation_position='top right')
fig.show()


## 7. Active vs inactive members


In [ ]:
g = df.groupby('IsActiveMember')['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
g.index = ['Inactive', 'Active']
print(g.reset_index().to_string(index=False))

fig = px.bar(g.reset_index(), x='index', y='churn_rate', text='churn_rate',
             title='Churn rate: active vs inactive members',
             labels={'index': '', 'churn_rate': 'Churn rate (%)'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()


## 8. Balance vs churn — the wealthy leave

Counterintuitive: churned customers hold a **higher** average balance (€91k vs €73k for retained).
The bank isn't losing dormant accounts — it's losing its most valuable ones.


In [ ]:
print('Avg balance — churned:  €{:,.0f}'.format(df.loc[df.Churn == 1, 'Balance'].mean()))
print('Avg balance — retained: €{:,.0f}'.format(df.loc[df.Churn == 0, 'Balance'].mean()))

fig = px.histogram(df, x='Balance', color=df['Churn'].map({0: 'Retained', 1: 'Exited'}),
                   nbins=40, barmode='overlay', opacity=0.6,
                   title='Balance distribution: exited vs retained',
                   labels={'color': 'Outcome'})
fig.show()


In [ ]:
# Balance deciles: churn climbs with balance (rank-based so zero balances don't break the bins)
df['BalDecile'] = pd.qcut(df['Balance'].rank(method='first'), 10,
                          labels=[f'D{i}' for i in range(1, 11)])
g = df.groupby('BalDecile', observed=True)['Churn'].agg(customers='count', churn_rate='mean')
g['churn_rate'] = (g['churn_rate'] * 100).round(2)
print(g.reset_index().to_string(index=False))

fig = px.bar(g.reset_index(), x='BalDecile', y='churn_rate', text='churn_rate',
             title='Churn rate by balance decile (D1 = lowest balance)',
             labels={'BalDecile': 'Balance decile', 'churn_rate': 'Churn rate (%)'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()


## 9. Tenure and credit score — mostly flat, no strong signal


In [ ]:
g = df.groupby('Tenure')['Churn'].mean().mul(100).round(2).reset_index()
print(g.to_string(index=False))
px.bar(g, x='Tenure', y='Churn', text='Churn', title='Churn rate by tenure (years)',
       labels={'Churn': 'Churn rate (%)'}).update_traces(texttemplate='%{text:.1f}%', textposition='outside').show()


In [ ]:
g = df.groupby('CreditBand', observed=True)['Churn'].mean().mul(100).round(2).reset_index()
print(g.to_string(index=False))
px.bar(g, x='CreditBand', y='Churn', text='Churn', title='Churn rate by credit-score band',
       labels={'Churn': 'Churn rate (%)', 'CreditBand': ''}).update_traces(texttemplate='%{text:.1f}%', textposition='outside').show()


## 10. Correlation with churn


In [ ]:
num = df.select_dtypes('number').drop(columns=['Churn'])
corr = num.corrwith(df['Churn']).sort_values(ascending=False).round(3)
print(corr.to_string())

fig = px.imshow(corr.to_frame().T, text_auto='.2f', aspect='auto',
                title='Correlation of numeric features with Churn', color_continuous_scale='RdBu_r')
fig.show()


## 11. The at-risk cohort — who to call first


In [ ]:
risk = df[(df.Geography == 'Germany') & (df.IsActiveMember == 0) & (df.Age.between(40, 60))]
r = len(risk)
rc = int(risk['Churn'].sum())  # churned customers INSIDE the cohort
print(f'Inactive Germans aged 40-60: {r} customers, churn rate {risk.Churn.mean()*100:.1f}%')
print(f'That is {rc/df.Churn.sum()*100:.1f}% of ALL churned customers in just {r/len(df)*100:.1f}% of the base.')


## Key insights

1. **Germany is the problem market** — 32.4% churn vs ~16% in France/Spain.
2. **Age 50–60 churns at 56%**; under-40s are largely loyal (<12%).
3. **2 products = loyalty sweet spot** (7.6% churn) → cross-sell 1-product customers (27.7% churn).
4. **Inactive members churn at 26.9%** vs 14.3% for active → re-engagement campaigns pay off.
5. **The wealthy leave**: churned customers average €91k balance vs €73k retained — retention must protect high-value accounts.
6. Credit score and tenure barely move churn — don't waste retention budget there.
7. Priority cohort: **inactive Germans aged 40–60 — 61.2% churn**, 17% of all exits.
